# COLMAP to 3DGS — traitement d'une vidéo sur Google Colab

Ce notebook restaure une compilation préconstruite de Spirula depuis Google Drive, récupère `colmap_to_gs` (branche `dev`), crée l'environnement Conda à partir de `environment/google.colab/conda_env.yml`, puis lance `scripts/videos_to_3dgs.sh`.

Les frames sont extraites par le script `videos_to_3dgs.sh` lui-même : aucune extraction séparée n'est effectuée dans le notebook.


In [7]:
import os

# ============================================================
# PARAMÈTRES À MODIFIER
# ============================================================

# Chemin absolu de la vidéo dans Google Drive
os.environ["VIDEO_DRIVE_PATH"] = "/content/drive/MyDrive/gsplat/input/IMG_5017.MOV"

# Nombre de frames à extraire
os.environ["NUM_FRAMES"] = "300"


# ============================================================
# PARAMÈTRES INTERNES
# ============================================================

os.environ["SPIRULA_ARCHIVE_DIR"] = "/content/drive/MyDrive/spirula-builds"
os.environ["PROJECT_DIR"] = "/content/colmap_to_gs"
os.environ["WORK_DIR"] = "/content/colmap_to_gs_work"
os.environ["DRIVE_OUTPUT_DIR"] = "/content/drive/MyDrive/colmap_to_gs_runs"
os.environ["GIT_BRANCH"] = "dev"


In [8]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
import shutil
import subprocess

video_path = os.environ["VIDEO_DRIVE_PATH"]
n_frames = int(os.environ["NUM_FRAMES"])

if n_frames <= 0:
    raise ValueError("NUM_FRAMES doit être strictement positif.")

if not os.path.isfile(video_path):
    raise FileNotFoundError(f"Vidéo introuvable : {video_path}")

os.makedirs(os.environ["WORK_DIR"], exist_ok=True)
os.makedirs(os.environ["DRIVE_OUTPUT_DIR"], exist_ok=True)

print("Vidéo :", video_path)
print("Frames à extraire :", n_frames)

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Aucun pilote NVIDIA détecté : active un runtime Colab avec GPU.")

subprocess.run(["nvidia-smi"], check=True)


Vidéo : /content/drive/MyDrive/gsplat/input/IMG_5017.MOV
Frames à extraire : 300


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [12]:
%%bash
set -Eeuo pipefail

export DEBIAN_FRONTEND=noninteractive

apt-get update

apt-get install -y --no-install-recommends \
    git \
    git-lfs \
    curl \
    zip \
    unzip \
    libvulkan1 \
    vulkan-tools

echo "[OK] Dépendances système installées."


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 27.0 kB in 1s (28.1 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
git is already the newest version (1:2.43.0-1ubuntu7.4).
curl is already the newest version (8.5.0-2ubuntu10.16).
zip is already the newest version (

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [13]:
import os
import subprocess
from pathlib import Path

# Installation de micromamba dans /content/bin.
os.makedirs("/content/bin", exist_ok=True)
micromamba_bin = "/content/bin/micromamba"

if not os.path.isfile(micromamba_bin):
    install_cmd = (
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj -C /content bin/micromamba"
    )
    subprocess.run(["bash", "-lc", install_cmd], check=True)

os.chmod(micromamba_bin, 0o755)

os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"
os.environ["PATH"] = "/content/bin:" + os.environ["PATH"]

print("Micromamba :", micromamba_bin)
subprocess.run([micromamba_bin, "--version"], check=True)


Micromamba : /content/bin/micromamba


CompletedProcess(args=['/content/bin/micromamba', '--version'], returncode=0)

In [14]:
import os
import shutil
import subprocess

repo = os.environ["PROJECT_DIR"]
branch = os.environ["GIT_BRANCH"]
url = "https://github.com/NicoIGN/colmap_to_gs.git"

if os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run(["git", "-C", repo, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", repo, "checkout", branch], check=True)
    subprocess.run(
        ["git", "-C", repo, "pull", "--ff-only", "origin", branch],
        check=True
    )
else:
    if os.path.exists(repo):
        shutil.rmtree(repo)
    subprocess.run(
        ["git", "clone", "--branch", branch, "--single-branch", url, repo],
        check=True
    )

subprocess.run(
    ["git", "-C", repo, "submodule", "update", "--init", "--recursive"],
    check=True
)

subprocess.run(["git", "-C", repo, "status", "--short", "--branch"], check=True)
subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], check=True)

env_file = os.path.join(repo, "environment", "google.colab", "conda_env.yml")
if not os.path.isfile(env_file):
    raise FileNotFoundError(f"Fichier d'environnement absent : {env_file}")

script = os.path.join(repo, "scripts", "videos_to_3dgs.sh")
if not os.path.isfile(script):
    raise FileNotFoundError(f"Script absent : {script}")

print("Fichier Conda :", env_file)
print("Script vidéo :", script)


Fichier Conda : /content/colmap_to_gs/environment/google.colab/conda_env.yml
Script vidéo : /content/colmap_to_gs/scripts/videos_to_3dgs.sh


In [ ]:
import os
import subprocess

repo = os.environ["PROJECT_DIR"]
env_file = os.path.join(
    repo, "environment", "google.colab", "conda_env.yml"
)

micromamba = "/content/bin/micromamba"
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"

# Création de l'environnement exactement à partir du YAML du dépôt.
# Le nom de l'environnement est défini dans le YAML : gsplat.
subprocess.run(
    [micromamba, "create", "--yes", "--file", env_file],
    check=True
)

# Vérification des composants principaux de l'environnement.
check_code = r'''
import sys
import torch
import numpy
import cv2
import tqdm

print("Python :", sys.version)
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print("NumPy :", numpy.__version__)
print("OpenCV :", cv2.__version__)
print("tqdm :", tqdm.__version__)
'''

subprocess.run(
    [micromamba, "run", "-n", "gsplat", "python", "-c", check_code],
    check=True
)


In [ ]:
import glob
import os
import shutil
import zipfile
from pathlib import Path

archive_dir = os.environ["SPIRULA_ARCHIVE_DIR"]

archives = sorted(
    glob.glob(os.path.join(archive_dir, "*.zip")),
    key=os.path.getmtime,
    reverse=True
)

if not archives:
    raise FileNotFoundError(f"Aucune archive ZIP dans {archive_dir}")

archive = archives[0]
extract_root = "/content/spirula_archive_extracted"
spirula_root = "/content/spirula-studio"

print("Archive sélectionnée :", archive)

shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(spirula_root, ignore_errors=True)
os.makedirs(extract_root, exist_ok=True)

with zipfile.ZipFile(archive) as zf:
    zf.extractall(extract_root)

# Recherche récursive du répertoire qui contient un binaire build*/spirula.
candidates = glob.glob(
    os.path.join(extract_root, "**", "build*", "spirula"),
    recursive=True
)
bins = [
    path for path in candidates
    if os.path.isfile(path)
]

if not bins:
    raise FileNotFoundError(
        "Aucun binaire build*/spirula trouvé dans l'archive."
    )

spirula_bin_source = sorted(bins)[0]
source_root = str(Path(spirula_bin_source).parent.parent)

# On recopie la racine de compilation afin de conserver les ressources
# et bibliothèques éventuellement présentes à côté du binaire.
shutil.copytree(source_root, spirula_root)

relative_bin = os.path.relpath(spirula_bin_source, source_root)
spirula_bin = os.path.join(spirula_root, relative_bin)

if not os.path.isfile(spirula_bin):
    raise FileNotFoundError(f"Binaire Spirula introuvable après copie : {spirula_bin}")

os.chmod(spirula_bin, os.stat(spirula_bin).st_mode | 0o111)

os.environ["SPIRULA_ROOT"] = spirula_root
os.environ["SPIRULA_BIN"] = spirula_bin

print("SPIRULA_ROOT =", spirula_root)
print("SPIRULA_BIN  =", spirula_bin)
print("Taille binaire :", os.path.getsize(spirula_bin), "octets")


In [ ]:
import os
import pathlib

work = os.environ["WORK_DIR"]
frames = os.path.join(work, "frames")
output = os.path.join(os.environ["DRIVE_OUTPUT_DIR"], "video_run")

os.makedirs(work, exist_ok=True)
os.makedirs(frames, exist_ok=True)
os.makedirs(output, exist_ok=True)

config_path = os.path.join(work, "colab_video_config.sh")

config = """#!/usr/bin/env bash
set -Eeuo pipefail

SPIRULA_ROOT="{spirula_root}"
SPIRULA_BIN="{spirula_bin}"
PYTHON_BIN="python3"

SFM_IMAGES_DIR="{frames}"
SFM_IMAGES_RECURSIVE="true"
OUTPUT_DIR="{output}"

COLMAP_METHOD="spirula"
TRAIN_BACKEND="spirula"

# Masquage du ciel
SFM_MASKS="on"
TRAIN_MASKS="on"
TRAIN_MASKS_BACKEND="spirula"
TRAIN_MASKS_LABELS="sky,clouds"
TRAIN_MASKS_MODE="empty"
TRAIN_MASKS_SPIRULA_DEVICE="auto"
MASKS_JOBS="1"
MASKS_ACCEPT_LICENSE="false"

export SS_NN_LOG=1

# Paramètres SfM
SFM_MAX_IMAGE_SIZE="2048"
SFM_ALIKED_MAX_FEATURES="4096"
SFM_THREADS="6"
SFM_SP_QUALITY="medium"
SFM_SP_PAIRS="prefilter"

# Découpage et entraînement
SPLIT_NUM_BLOCKS="2"
SPLIT_OVERLAP_MAX_FRACTION="0.10"

TRAIN_NUM_ITER="12000"
TRAIN_SP_PRESET="3dgs"
TRAIN_SP_QUALITY="high"
TRAIN_SP_CAP_MAX="500000"
TRAIN_SP_REFINE_STOP_ITER="10000"
TRAIN_SP_REFINE_STOP_NUM_ITER="2000"
TRAIN_SP_FLOATER_SUPPRESSION="mild"
TRAIN_CLEAN="false"

# Post-traitement et export
POSTPROCESS_SPARSE_BBOX_MODE="apply"
POSTPROCESS_MAX_REMOVAL_FRACTION="0.20"

EXPORT_SPLAT_NAME="Video_20260515_1755_16_064_3dgs.splat"
""".format(
    spirula_root=os.environ["SPIRULA_ROOT"],
    spirula_bin=os.environ["SPIRULA_BIN"],
    frames=frames,
    output=output
)

pathlib.Path(config_path).write_text(config, encoding="utf-8")
os.chmod(config_path, 0o755)

print("Configuration :", config_path)
print(config)


In [ ]:
import os
import shlex
import subprocess

repo = os.environ["PROJECT_DIR"]
work = os.environ["WORK_DIR"]

config_path = os.path.join(work, "colab_video_config.sh")
frames = os.path.join(work, "frames")
log_path = os.path.join(work, "videos_to_3dgs.log")

cmd = [
    "/content/bin/micromamba",
    "run", "-n", "gsplat",
    "bash",
    "scripts/videos_to_3dgs.sh",
    config_path,
    "--video", os.environ["VIDEO_DRIVE_PATH"],
    "--num-frames", os.environ["NUM_FRAMES"],
    "--frames-dir", frames
]

print("Commande :", shlex.join(cmd), flush=True)
print("Répertoire courant :", repo, flush=True)
print("Log :", log_path, flush=True)

with open(log_path, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        cwd=repo,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    status = proc.wait()

if status != 0:
    raise subprocess.CalledProcessError(status, cmd)

print("Pipeline terminé avec succès.")


In [ ]:
import os

output = os.path.join(
    os.environ["DRIVE_OUTPUT_DIR"],
    "video_run"
)

if not os.path.isdir(output):
    raise FileNotFoundError(f"Dossier de sortie absent : {output}")

found = 0
for root, _, files in os.walk(output):
    for name in files:
        path = os.path.join(root, name)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{size_mb:9.2f} MiB  {path}")
        found += 1

if found == 0:
    print("Aucun fichier de sortie trouvé.")
